In [1]:
import blocks as layers
import torch
import torch.nn as nn

/home/cybertesla/Transformers/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
class Transformer(nn.Module):

    def __init__(self, src_vocab_size: int, tgt_vocab_size: int, d_model: int, num_heads: int, hidden_dim: int, num_blocks: int, dropout_rate: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.encoder = layers.EncoderStack(src_vocab_size, d_model, num_heads, hidden_dim, num_blocks, dropout_rate)
        self.decoder = layers.DecoderStack(tgt_vocab_size, d_model, num_heads, hidden_dim, num_blocks, dropout_rate)
        self.final_linear_layer = nn.Linear(d_model, tgt_vocab_size)


    def forward(self, src_tokens: torch.Tensor, tgt_tokens: torch.Tensor):
        
        encoder_output = self.encoder(src_tokens)
        decoder_output = self.decoder(tgt_tokens, encoder_output)
        logits = self.final_linear_layer(decoder_output)

        return logits

In [3]:
batch_size = 2
num_src_tokens = 3
num_tgt_tokens = 5
d_model = 8
num_heads = 2
head_dim = 4
src_vocab_size = 5
tgt_vocab_size = 7
hidden_dim = 16
num_blocks = 6

In [4]:
dummy_src_input = torch.randint(low=0, high=src_vocab_size, size=(batch_size, num_src_tokens))
dummy_tgt_input = torch.randint(low=0, high=tgt_vocab_size, size=(batch_size, num_tgt_tokens))
print(dummy_src_input.shape)
print(dummy_tgt_input.shape)
print("(batch_size, num_tokens)")

torch.Size([2, 3])
torch.Size([2, 5])
(batch_size, num_tokens)


In [5]:
transformer =Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, hidden_dim, num_blocks)
transformer

Transformer(
  (encoder): EncoderStack(
    (embedding): Embedding(5, 8)
    (positional_encoding): PositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (attention_layer): MultiHeadAttention(
          (w_q): Linear(in_features=8, out_features=8, bias=False)
          (w_k): Linear(in_features=8, out_features=8, bias=False)
          (w_v): Linear(in_features=8, out_features=8, bias=False)
          (w_o): Linear(in_features=8, out_features=8, bias=False)
        )
        (ffnn): FFNN(
          (hidden_layer): Linear(in_features=8, out_features=16, bias=True)
          (output_layer): Linear(in_features=16, out_features=8, bias=True)
          (activation): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (norm_1): LayerNorm((8,), 

In [6]:
transformer_output = transformer(dummy_src_input, dummy_tgt_input)
transformer_output

tensor([[[-0.5239,  0.2281,  0.4817, -0.6046, -0.6842,  0.2576, -0.3847],
         [-0.6369,  0.4949,  0.8515, -0.4595, -0.7206,  0.3878, -0.3391],
         [ 0.7332,  0.6091, -0.9446, -0.1164, -0.4975,  0.4703,  0.1482],
         [-0.6516,  0.4036,  0.7171, -1.1917,  0.7395, -0.3290, -0.0877],
         [ 0.4453, -0.5619, -0.6247,  0.5121, -1.0040,  0.9128,  0.4330]],

        [[-0.5770,  0.0559,  0.6085, -0.7046, -0.5090,  0.2133, -0.3481],
         [-0.6316,  0.4667,  0.0078,  0.1605,  0.3567, -0.0547, -0.2563],
         [-0.2382,  0.3531,  0.3606, -0.0317, -1.0348,  0.7522, -0.3340],
         [ 0.0995,  0.2986, -0.6274,  0.0331,  0.4172,  0.2231, -0.0268],
         [-0.5925,  0.6640,  0.8177,  0.6533, -1.0266,  0.8573, -0.4044]]],
       grad_fn=<ViewBackward0>)